In [ ]:
import os
os.chdir(os.path.join(os.path.dirname("__file__"), ".."))
import sys
sys.path.insert(0, "..")
import matplotlib.pyplot as plt
from src.data.loaders import load_iam, load_funsd
from src.data.synthetic import generate_dataset
from src.data.preprocessing import preprocess
from src.ocr.trocr_finetune import finetune, load_trocr
from src.ocr.baseline import extract_text as tesseract_extract
from src.ocr.evaluate import compute_metrics, compare_models

In [ ]:
# Run on Kaggle GPU â€” uses full data caps from config
iam_train, iam_val = load_iam(max_train=2000, max_val=500)
funsd_train, funsd_test = load_funsd()
synthetic_docs = generate_dataset(n=1500, seed=42)

train_docs = iam_train + funsd_train + synthetic_docs[:1200]
val_docs = iam_val + funsd_test + synthetic_docs[1200:]
print(f"Train: {len(train_docs)}, Val: {len(val_docs)}")

In [ ]:
# Fine-tune TrOCR (~2-3 hours on Kaggle T4 GPU)
model_dir = finetune(train_docs, val_docs)
print(f"Model saved to: {model_dir}")

In [ ]:
_, test_docs = load_iam(max_train=10, max_val=200)
references = [doc.text for doc in test_docs]

tesseract_results = [tesseract_extract(preprocess(doc.image)) for doc in test_docs]
trocr_fn = load_trocr(model_dir)
trocr_results = [trocr_fn(preprocess(doc.image)) for doc in test_docs]

comparison = compare_models(tesseract_results, trocr_results, references)
print(f"Tesseract  CER: {comparison['tesseract']['cer']:.4f}  WER: {comparison['tesseract']['wer']:.4f}")
print(f"TrOCR      CER: {comparison['trocr']['cer']:.4f}  WER: {comparison['trocr']['wer']:.4f}")

In [ ]:
labels = ["CER", "WER"]
tesseract_vals = [comparison["tesseract"]["cer"], comparison["tesseract"]["wer"]]
trocr_vals = [comparison["trocr"]["cer"], comparison["trocr"]["wer"]]
x = range(len(labels))
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([i - 0.2 for i in x], tesseract_vals, 0.4, label="Tesseract", color="#FF6B6B")
ax.bar([i + 0.2 for i in x], trocr_vals, 0.4, label="Fine-tuned TrOCR", color="#4ECDC4")
ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_title("Tesseract vs Fine-tuned TrOCR")
ax.legend()
plt.show()